# Interview MCQ Trainer

Добро пожаловать в **Interview MCQ Trainer** - интерактивный Jupyter-ноутбук для подготовки к техническим собеседованиям в Data Scientists, ML Engineers, Data Analysts.  

## 📌 Как это работает?

- Вы выбираете тему или проходите вопросы в случайном порядке  
- Для каждого вопроса предлагаются 4 варианта ответа, из которых только один верный  
- После выбора ответа правильного ответа, вы получаете пояснение и отдельную ссылку на ресурс, где подробно объясняется эта тема
- Вопросы, на которые даны неверные ответы, можно повторно изучить позже воспользовавшись статистикой по ответам 

## 🎯 Что включает тренировка?

1️⃣ **Реалистичные вопросы** - собраны с учётом требований к позициям DS/ML  
2️⃣ **Разные уровни сложности** - от базовых тем Python и Math до продвинутых ML-методов  
3️⃣ **Поддержка вопросов ссылками с пояснениями и статистикой по ответам** - к каждому вопросу выводится ссылка на ресурс с пояснением на русском, а также после ответов на вопросы выводиться статистика по ответам, что позволяет увидеть вопросы по которым есть сложности и повторить их отдельно

## 🚀 Как использовать?

1. Убедитесь, что у вас установлен **pandas** и Jupyter  
2. Запустите блоки ниже, чтобы загрузить CSV-файлы с вопросами  
3. Выберите тему, отвечайте и получайте фидбек  

Готовы? Жмите "Ctrl + Enter" и начинаем! 🚀

## Инициализация библиотек и загрузка вопросов из CSV-файлов

In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML, clear_output

# # Настройки pandas
# pd.set_option('display.max_colwidth', None)


In [10]:
# Загрузка данных из CSV
topics_df = pd.read_csv("./data/topics.csv")
questions_df = pd.read_csv("./data/questions.csv")
answers_df = pd.read_csv("./data/answers.csv")
statistics_df = pd.read_csv("./data/user_statistics.csv", parse_dates=["last_attempt"])


# Выводим короткую статистику
print(f"📁 Загружено тем: {len(topics_df)}")
print(f"❓ Загружено вопросов: {len(questions_df)}")
print(f"🅰️ Загружено ответов: {len(answers_df)}")
print(f"📊 Загружено статистик: {len(statistics_df)}")

📁 Загружено тем: 18
❓ Загружено вопросов: 46
🅰️ Загружено ответов: 184
📊 Загружено статистик: 42


## Вспомогательные функции

In [3]:
def get_topics():
    return topics_df[['topic_id', 'name']]

def get_questions(topic_id=-1, min_difficulty=-1):
    df = questions_df.copy()
    if topic_id != -1:
        df = df[df['topic_id'] == topic_id]
    if min_difficulty != -1:
        df = df[df['difficulty_level'] >= min_difficulty]
    return df[['question_id', 'question_text', 'link']]

def check_answer(answer_id):
    answer = answers_df[answers_df['answer_id'] == int(answer_id)]
    if answer.empty:
        return False
    return answer['is_correct'].iloc[0]

In [4]:
def get_topics():
    # Получает список всех тем из загруженного файла.
    # Возвращает DataFrame с колонками topic_id и name.
    return topics_df[['topic_id', 'name']]

def get_questions(topic_id=-1, min_difficulty=-1):
    """
    Получает вопросы по теме и минимальной сложности.
    Если topic_id == -1 и min_difficulty == -1, возвращает все вопросы.
    """
    df = questions_df.copy()
    if topic_id != -1:
        df = df[df['topic_id'] == topic_id]
    if min_difficulty != -1:
        df = df[df['difficulty_level'] >= min_difficulty]
    return df[['question_id', 'question_text', 'link']]

def check_answer(answer_id):
    # Проверяет, является ли ответ с заданным ID правильным.
    # Возвращает булево значение: True — если правильный, False — если нет.
    answer = answers_df[answers_df['answer_id'] == int(answer_id)]
    return not answer.empty and answer['is_correct'].iloc[0]

# Локальная "статистика" на время сессии (в памяти)
user_statistics = {}

def update_statistics(user_id, question_id, is_correct):
    global statistics_df
    now = pd.Timestamp.now()
    mask = (statistics_df['user_id'] == user_id) & (statistics_df['question_id'] == question_id)
    if not statistics_df[mask].empty:
        idx = statistics_df[mask].index[0]
        statistics_df.at[idx, 'attempts'] += 1
        statistics_df.at[idx, 'correct_attempts'] += int(is_correct)
        statistics_df.at[idx, 'incorrect_attempts'] += int(not is_correct)
        statistics_df.at[idx, 'last_attempt'] = now
    else:
        statistics_df.loc[len(statistics_df)] = [
            user_id, question_id, 1,
            int(is_correct), int(not is_correct), now
        ]

def show_analytics(user_id):
    """
    Возвращает агрегированную статистику по темам и сложности на основе statistics_df.
    """
    # Отбираем только строки для данного пользователя
    user_df = statistics_df[statistics_df['user_id'] == user_id]
    if user_df.empty:
        return pd.DataFrame()

    # Присоединяем информацию о вопросах (topic_id)
    merged = user_df.merge(questions_df[['question_id', 'topic_id', 'difficulty_level']], on='question_id', how='left')

    # Присоединяем информацию о темах (названия)
    merged = merged.merge(topics_df[['topic_id', 'name']], on='topic_id', how='left')

    # Агрегация
    summary = merged.groupby(['name', 'difficulty_level']).agg(
        Всего=('attempts', 'sum'),
        Верно=('correct_attempts', 'sum'),
        Неверно=('incorrect_attempts', 'sum')
    ).reset_index()

    # Переименуем для удобства в display_analytics (или можно переименовывать там)
    summary = summary.rename(columns={'name': 'Тема', 'difficulty_level': 'Сложность'})
    return summary


def auto_height(text, line_height=16, min_height=40, max_height=400, wrap_limit=100):
    """
    Вычисляет адекватную высоту Textarea по содержимому, включая переносы строк.
    wrap_limit — кол-во символов в строке до визуального переноса.
    """
    lines = text.split('\n')
    estimated_lines = sum((len(line) // wrap_limit + 1) for line in lines)
    height = min(max(estimated_lines * line_height, min_height), max_height)
    return widgets.Layout(width='100%', height=f'{height}px')

def md_output(markdown_text, min_height=80):
    box = widgets.Output(layout=widgets.Layout(
        width='100%', min_height=f'{min_height}px',
        overflow_y='auto', border='1px solid #ddd', padding='6px'
    ))
    with box:
        if "```" in markdown_text:
            clean = markdown_text.replace("```python", "").replace("```", "")
            # HTML-блок: белый фон, чёрный текст, моноширинный шрифт

            display(HTML(
                f"<pre style='background-color:#fff !important; color:#000 !important; "
                f"padding:4px; margin:0; margin-top:-4px; font-size:13px; "
                f"line-height:1.2; font-family:monospace;'>"
                f"<code style='background-color:#fff !important; color:#000 !important; "
                f"margin:0; font-family:monospace;'>{clean}</code></pre>"
            ))
        else:
            display(Markdown(markdown_text))
    return box


In [5]:
def create_settings_interface():
    """
    Создаёт интерактивный интерфейс настроек с помощью виджетов Jupyter.

    Функция позволяет пользователю выбрать:
    - Тему из списка CSV (или 'Все темы');
    - Интервал повторения вопроса, если на него был дан неверный ответ (10, 30 или 60 минут);
    - Уровень сложности вопроса (от 1 до 5, либо 'Любая сложность').

    После нажатия кнопки "Сохранить настройки", выбранные значения сохраняются
    в глобальные переменные: selected_topic, selected_interval, selected_difficulty,
    и отображается сообщение о том, что настройки успешно сохранены.
    """

    # Получение списка тем из topics_df
    topics = get_topics()
    topic_options = [('Все темы', -1)] + [(row['name'], row['topic_id']) for _, row in topics.iterrows()]

    # Виджет выбора темы
    topic_dropdown = widgets.Dropdown(
        options=topic_options,
        description='Тема:',
    )

    # Виджет выбора интервала (для повторения ошибочных вопросов)
    interval_dropdown = widgets.Dropdown(
        options=[('10 минут', 10), ('30 минут', 30), ('60 минут', 60)],
        description='Интервал:',
    )

    # Пояснение к интервалу
    interval_help = widgets.HTML(
        "<span style='font-size: 12px; color: gray;'>Интервал повторения вопроса при ошибке</span>"
    )

    # Объединяем интервал и пояснение в один блок
    interval_block = widgets.HBox([interval_dropdown, interval_help])

    # Виджет выбора уровня сложности
    difficulty_dropdown = widgets.Dropdown(
        options=[
            ('Любая сложность', -1),
            ('1 - Лёгкие', 1),
            ('2 - Ниже среднего', 2),
            ('3 - Средние', 3),
            ('4 - Выше среднего', 4),
            ('5 - Сложные', 5)
        ],
        description='Сложность:',
    )

    # Кнопка для сохранения настроек
    save_button = widgets.Button(
        description='Сохранить настройки',
        button_style='success'
    )

    output = widgets.Output()

    # Глобальные переменные для хранения настроек
    global selected_topic, selected_interval, selected_difficulty
    selected_topic = None
    selected_interval = None
    selected_difficulty = None

    # Обработчик нажатия кнопки
    def on_save_button_clicked(b):
        global selected_topic, selected_interval, selected_difficulty
        selected_topic = topic_dropdown.value
        selected_interval = interval_dropdown.value
        selected_difficulty = difficulty_dropdown.value
        with output:
            output.clear_output()
            print(
                f"Настройки сохранены: Тема - {topic_dropdown.label}, "
                f"Интервал - {interval_dropdown.label} минут, "
                f"Сложность - {difficulty_dropdown.label}"
            )
            display(widgets.HTML("<b>✅ Настройки сохранены. Вы можете перейти к следующему шагу.</b>"))

    save_button.on_click(on_save_button_clicked)

    # Отображение всех виджетов
    display(topic_dropdown, interval_block, difficulty_dropdown, save_button, output)


In [6]:
current_question_index = 0

def display_question(topic_id, min_difficulty):
    """
    Функция отображает вопрос и ответы в специальном виджете для выбранной темы.
    Использует pandas-таблицы и сохраняет статистику в глобальный DataFrame.
    """
    global current_question_index, statistics_df

    questions = get_questions(topic_id, min_difficulty)
    print(f"📄 Найдено вопросов: {len(questions)}")
    if questions.empty:
        display(widgets.HTML("<b>⚠️ Нет доступных вопросов для выбранной темы.</b>"))
        return

    # # Если есть колонка created_at — сортируем, иначе нет
    # if 'created_at' in questions.columns:
    questions = questions.sort_values(by='question_id', ascending=False)
    # else:
    #     questions = questions.sample(frac=1).reset_index(drop=True)

    if current_question_index >= len(questions):
        display(widgets.HTML("<b>✅ Все вопросы пройдены.</b>"))
        return

    question = questions.iloc[current_question_index]
    question_id = question['question_id']
    question_text = question['question_text']
    question_link = question.get('link', '')

    # Получение всех ответов к вопросу
    answers = answers_df[answers_df['question_id'] == question_id][['answer_id', 'answer_text']]
    if answers.empty:
        display(widgets.HTML("<b>⚠️ Нет ответов к данному вопросу.</b>"))
        return

    answers = answers.sample(frac=1).reset_index(drop=True)  # перемешиваем

    # Отображение вопроса
    question_textarea = md_output(question_text, min_height=50)

    # Подготовка ответов
    answer_textareas = [
        widgets.Textarea(value=row['answer_text'], layout=auto_height(row['answer_text']))
        for _, row in answers.iterrows()
    ]
    answer_buttons = [
        widgets.Button(description='Выбрать', layout=widgets.Layout(width='20%'))
        for _ in range(len(answers))
    ]

    output = widgets.Output()

    def on_button_click(b):
        selected_index = answer_buttons.index(b)
        selected_answer_id = answers.iloc[selected_index]['answer_id']
        is_correct = check_answer(selected_answer_id)

        # Обновляем статистику в DataFrame
        now = pd.Timestamp.now()
        user_id = 'user_id'  # можно заменить на session_id

        mask = (statistics_df['user_id'] == user_id) & (statistics_df['question_id'] == question_id)
        if not statistics_df[mask].empty:
            idx = statistics_df[mask].index[0]
            statistics_df.at[idx, 'attempts'] += 1
            statistics_df.at[idx, 'correct_attempts'] += int(is_correct)
            statistics_df.at[idx, 'incorrect_attempts'] += int(not is_correct)
            statistics_df.at[idx, 'last_attempt'] = now
        else:
            statistics_df.loc[len(statistics_df)] = [
                user_id, question_id, 1,
                int(is_correct), int(not is_correct), now
            ]

        with output:
            output.clear_output()
            display(widgets.HTML(
                f"<b>{'✅ Правильно!' if is_correct else '❌ Неправильно!'}</b> " +
                (f"<a href='{question_link}' target='_blank'>Ссылка на доку</a>" if question_link else "")
            ))

    for button in answer_buttons:
        button.on_click(on_button_click)

    # Кнопка "Следующий вопрос"
    next_question_button = widgets.Button(
        description='Следующий вопрос',
        button_style='info'
    )

    def on_next_question_button_clicked(_):
        global current_question_index
        current_question_index += 1
        output.clear_output()
        display_question(topic_id, min_difficulty)

    next_question_button.on_click(on_next_question_button_clicked)

    # Кнопка "Завершить"
    finish_button = widgets.Button(
        description='Завершить',
        button_style='danger'
    )

    def on_finish_button_clicked(_):
        with output:
            output.clear_output()
            display(widgets.HTML("<b>🎓 Вы завершили текущую сессию. Перейдите к аналитике или повторению.</b>"))

    finish_button.on_click(on_finish_button_clicked)

    # Отображение интерфейса
    display(question_textarea)
    for textarea, button in zip(answer_textareas, answer_buttons):
        display(widgets.HBox([textarea, button]))
    display(widgets.HBox([next_question_button, finish_button]))
    display(output)


In [7]:
def display_analytics(user_id):
    """
    Отображает подробную статистику по темам и сложности.
    Анализ строится по таблице statistics_df, с привязкой к темам и уровню сложности.
    """
    # Получаем аналитику в формате DataFrame
    analytics = show_analytics(user_id)

    if analytics.empty:
        display(widgets.HTML("<b>❕ Нет данных по пользователю.</b>"))
        return

    # Переименуем колонки для отображения
    analytics = analytics.rename(columns={
        'Всего': 'Всего ответов',
        'Верно': 'Правильные',
        'Неверно': 'Ошибки'
    })

    display(widgets.HTML("<h4>📊 Подробная статистика по темам</h4>"))
    display(analytics)

    # Фильтрация слабых мест: точность < 50%
    weak_rows = analytics[
        (analytics['Правильные'] / analytics['Всего ответов']) < 0.5
    ]

    if not weak_rows.empty:
        display(widgets.HTML("<h4>⚠️ Слабые места:</h4>"))
        for _, row in weak_rows.iterrows():
            display(widgets.HTML(
                f"- {row['Тема']}: "
                f"{row['Правильные']} из {row['Всего ответов']}, "
                f"Ошибок: {row['Ошибки']}"
            ))


In [8]:
create_settings_interface()

Dropdown(description='Тема:', options=(('Все темы', -1), ('Python', 1), ('NumPy', 2), ('Pandas', 3), ('EDA (Ex…

Dropdown(description='Сложность:', options=(('Любая сложность', -1), ('1 - Лёгкие', 1), ('2 - Ниже среднего', …

Button(button_style='success', description='Сохранить настройки', style=ButtonStyle())

Output()

In [11]:
display_question(selected_topic, selected_difficulty)

📄 Найдено вопросов: 32


Output(layout=Layout(border='1px solid #ddd', min_height='50px', overflow_y='auto', padding='6px', width='100%…

Output()

Output(layout=Layout(border='1px solid #ddd', min_height='50px', overflow_y='auto', padding='6px', width='100%…

Output()

In [10]:
statistics_df

,user_id,question_id,attempts,correct_attempts,incorrect_attempts,last_attempt
0,user_id,32,3,2,1,2025-09-17 12:54:43.468244
1,user_id,31,2,2,0,2025-09-17 12:56:18.425745
2,user_id,30,5,2,3,2025-09-17 13:01:47.920349
3,user_id,29,2,2,0,2025-09-17 13:12:10.855804
4,user_id,36,3,3,0,2025-09-16 11:09:45.587071
5,user_id,35,3,3,0,2025-09-16 11:10:03.132441
6,user_id,34,4,3,1,2025-09-16 11:10:16.535187
7,user_id,33,3,3,0,2025-09-16 11:10:26.304648
8,user_id,43,1,1,0,2025-09-16 10:28:31.292939
9,user_id,42,1,1,0,2025-09-16 10:59:13.881753


In [11]:
display_analytics('user_id')

HTML(value='<h4>📊 Подробная статистика по темам</h4>')

,Тема,Сложность,Всего ответов,Правильные,Ошибки
0,Math,1,10,9,1
1,Math,2,7,7,0
2,Math,3,3,2,1
3,Python,1,4,4,0
4,Python,2,5,4,1
5,Python,3,25,21,4
6,Python,4,8,5,3
7,Python,5,2,2,0


In [12]:
import csv
import os

def save_df_to_csv(df, path):
    df.to_csv(
        path,
        index=False,
        quoting=csv.QUOTE_ALL,
        quotechar='"',
        lineterminator='\n',
        encoding="utf-8"
    )

save_df_to_csv(statistics_df, os.path.join("./data", "user_statistics.csv"))